# Ensemble Learning from scratch
Random Forest, AdaBoost und Gradient Boosting – nur mit `DecisionTree` aus sklearn.

In [1]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.datasets import make_classification, make_friedman1
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=2000, n_features=20, n_informative=8, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, random_state=0)
acc = lambda m: (m.predict(Xte) == yte).mean()

## Random Forest
Bootstrap-Sample pro Baum, zufällige Feature-Auswahl pro Split (`max_features`), Mehrheitsvotum.

In [2]:
class RandomForest:
    def __init__(self, n=100):
        self.n = n

    def fit(self, X, y):
        rng = np.random.default_rng(0)
        self.trees = []
        for _ in range(self.n):
            idx = rng.integers(0, len(X), len(X))  # Ziehen mit Zurücklegen
            self.trees.append(DecisionTreeClassifier(max_features="sqrt").fit(X[idx], y[idx]))
        return self

    def predict(self, X):
        votes = np.array([t.predict(X) for t in self.trees])
        return (votes.mean(axis=0) > 0.5).astype(int)

print("Einzelner Baum:", acc(DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)))
print("Random Forest: ", acc(RandomForest().fit(Xtr, ytr)))

Einzelner Baum: 0.866
Random Forest:  0.92


## AdaBoost
Tree Stumps auf gewichteten Daten, $\alpha_t = \frac{1}{2}\ln\frac{1-\varepsilon_t}{\varepsilon_t}$, Vorhersage $H(x)=\text{sign}\left(\sum_t \alpha_t h_t(x)\right)$.

In [3]:
class AdaBoost:
    def __init__(self, n=200):
        self.n = n

    def fit(self, X, y):
        y = 2 * y - 1                          # Labels {0,1} -> {-1,+1}
        w = np.full(len(X), 1 / len(X))
        self.stumps, self.alphas = [], []
        for _ in range(self.n):
            h = DecisionTreeClassifier(max_depth=1).fit(X, y, sample_weight=w)
            pred = h.predict(X)
            eps = np.clip(w[pred != y].sum(), 1e-10, 1 - 1e-10)
            alpha = 0.5 * np.log((1 - eps) / eps)
            w *= np.exp(-alpha * y * pred)     # Fehler hoch-, Treffer heruntergewichten
            w /= w.sum()
            self.stumps.append(h); self.alphas.append(alpha)
        return self

    def predict(self, X):
        score = sum(a * h.predict(X) for a, h in zip(self.alphas, self.stumps))
        return (score > 0).astype(int)

print("Einzelner Stump:", acc(DecisionTreeClassifier(max_depth=1).fit(Xtr, ytr)))
print("AdaBoost:       ", acc(AdaBoost().fit(Xtr, ytr)))

Einzelner Stump: 0.716
AdaBoost:        0.898


## Gradient Boosting (Regression)
Jeder Baum lernt den Restfehler: $r \leftarrow r - \eta \cdot f_t(X)$, Vorhersage $\bar y + \eta \sum_t f_t(x)$.

In [4]:
class GBT:
    def __init__(self, n=200, lr=0.1, max_depth=3):
        self.n, self.lr, self.max_depth = n, lr, max_depth

    def fit(self, X, y):
        self.f0 = y.mean()
        r = y - self.f0                        # Restfehler
        self.trees = []
        for _ in range(self.n):
            f = DecisionTreeRegressor(max_depth=self.max_depth).fit(X, r)
            r = r - self.lr * f.predict(X)
            self.trees.append(f)
        return self

    def predict(self, X):
        return self.f0 + self.lr * sum(f.predict(X) for f in self.trees)

Xr, yr = make_friedman1(n_samples=2000, noise=1.0, random_state=0)
Xrtr, Xrte, yrtr, yrte = train_test_split(Xr, yr, random_state=0)
mse = lambda m: ((m.predict(Xrte) - yrte) ** 2).mean()

print("Einzelner Baum (MSE):", mse(DecisionTreeRegressor(max_depth=3).fit(Xrtr, yrtr)))
print("GBT (MSE):           ", mse(GBT().fit(Xrtr, yrtr)))

Einzelner Baum (MSE): 10.668628399457491
GBT (MSE):            1.7971937230816821
